# 🧪 03. Statistical Hypothesis Testing & Effect Size
**Project**: E-Commerce Discount Impact Analysis & Profit Prediction  
**Role**: Senior Data Scientist & Statistician

---

## Overview & Hypothesis Formulation
We conduct a rigorous statistical hypothesis test to determine whether promotional discounts have a statistically and practically significant impact on Average Order Value (AOV).

$$\begin{cases}
H_0: \mu_{\text{discount}} = \mu_{\text{no\_discount}} \quad \text{(Null Hypothesis: Mean AOV is identical between groups)} \\
H_1: \mu_{\text{discount}} \neq \mu_{\text{no\_discount}} \quad \text{(Alternative Hypothesis: Mean AOV differs between groups)}
\end{cases}$$

- **Significance Level**: $\alpha = 0.05$ (two-sided).
- **Test Metric**: Independent two-sample **Welch's $t$-test** (robust to unequal variances).
- **Supplementary Metrics**: Mann-Whitney $U$ test, Welch 95% Confidence Interval, Cohen's $d$, and Hedges' $g$.


In [1]:
from typing import Dict, Any, Tuple
import numpy as np
import pandas as pd
from scipy import stats

orders_df = pd.read_csv('data/processed/orders_delivered_aggregated.csv')
disc_aov = orders_df.loc[orders_df['has_discount'], 'order_value'].dropna()
nodisc_aov = orders_df.loc[~orders_df['has_discount'], 'order_value'].dropna()

print(f"Delivered Orders: {len(orders_df):,}")
print(f"Discounted Sample (n1):     {len(disc_aov):,}")
print(f"Non-Discounted Sample (n2): {len(nodisc_aov):,}")

Delivered Orders: 4,386
Discounted Sample (n1):     4,230
Non-Discounted Sample (n2): 156


## Step 1: Statistical Assumption Checking

In [2]:
def check_normality(group1: pd.Series, group2: pd.Series) -> Dict[str, Any]:
    # Evaluates skewness, kurtosis, and D'Agostino normality test
    results = {}
    for grp, label in zip([group1, group2], ['Discounted', 'Non-Discounted']):
        dag_stat, dag_p = stats.normaltest(grp)
        results[label] = {
            'n': len(grp),
            'mean': float(grp.mean()),
            'std': float(grp.std()),
            'skewness': float(stats.skew(grp)),
            'kurtosis': float(stats.kurtosis(grp)),
            'dagostino_stat': float(dag_stat),
            'dagostino_p': float(dag_p)
        }
    return results

def check_variance_equality(group1: pd.Series, group2: pd.Series) -> Dict[str, Any]:
    # Tests homoscedasticity using Levene's test centered at median
    levene_stat, levene_p = stats.levene(group1, group2, center='median')
    return {
        'levene_stat': float(levene_stat),
        'levene_p': float(levene_p),
        'var_discounted': float(group1.var()),
        'var_nondiscounted': float(group2.var()),
        'variance_ratio': float(group1.var() / group2.var()),
        'equal_var_decision': bool(levene_p >= 0.05)
    }

norm_check = check_normality(disc_aov, nodisc_aov)
var_check = check_variance_equality(disc_aov, nodisc_aov)

print("=== NORMALITY ASSUMPTION CHECK ===")
display(pd.DataFrame(norm_check).T)

print("\n=== VARIANCE HOMOGENEITY CHECK (LEVENE'S TEST) ===")
for k, v in var_check.items():
    print(f"  {k:22s}: {v}")
print("\n-> Levene p < 0.05 confirms unequal variances; Welch's t-test is required.")

=== NORMALITY ASSUMPTION CHECK ===


,n,mean,std,skewness,kurtosis,dagostino_stat,dagostino_p
Discounted,4230.0,35550.653794,22755.743900,0.707622,0.252389,299.356141,9.900105e-66
Non-Discounted,156.0,18799.192308,14431.325187,1.087221,1.208394,29.537528,3.854849e-07



=== VARIANCE HOMOGENEITY CHECK (LEVENE'S TEST) ===
  levene_stat           : 37.99960953734541
  levene_p              : 7.712406812869787e-10
  var_discounted        : 517823880.4374478
  var_nondiscounted     : 208263146.64665017
  variance_ratio        : 2.4863922819528606
  equal_var_decision    : False

-> Levene p < 0.05 confirms unequal variances; Welch's t-test is required.


## Step 2: Independent Two-Sample Welch's $t$-Test

In [3]:
def perform_welch_ttest(g1: pd.Series, g2: pd.Series, alpha: float = 0.05) -> Dict[str, Any]:
    # Executes Welch's t-test and computes Satterthwaite degrees of freedom
    t_stat, p_val = stats.ttest_ind(g1, g2, equal_var=False, alternative='two-sided')
    n1, n2 = len(g1), len(g2)
    v1, v2 = g1.var(ddof=1), g2.var(ddof=1)
    dof = (v1 / n1 + v2 / n2)**2 / (((v1 / n1)**2 / (n1 - 1)) + ((v2 / n2)**2 / (n2 - 1)))
    return {
        't_statistic': float(t_stat),
        'p_value': float(p_val),
        'degrees_of_freedom': float(dof),
        'alpha': alpha,
        'decision': "Reject H0" if p_val < alpha else "Fail to reject H0"
    }

welch_res = perform_welch_ttest(disc_aov, nodisc_aov)
for k, v in welch_res.items():
    print(f"  {k:22s}: {v}")

  t_statistic           : 13.875792073637808
  p_value               : 1.866502282488147e-30
  degrees_of_freedom    : 184.67236156483705
  alpha                 : 0.05
  decision              : Reject H0


## Step 3: Non-Parametric Robustness (Mann-Whitney $U$)

In [4]:
mwu_stat, mwu_p = stats.mannwhitneyu(disc_aov, nodisc_aov, alternative='two-sided')
print(f"Mann-Whitney U Statistic: {mwu_stat:,.1f}")
print(f"Mann-Whitney p-value:     {mwu_p:.4e}")

Mann-Whitney U Statistic: 480,496.5
Mann-Whitney p-value:     3.2190e-22


## Step 4: Effect Size and 95% Confidence Interval

In [5]:
def compute_effect_sizes_and_ci(g1: pd.Series, g2: pd.Series, alpha: float = 0.05) -> Dict[str, Any]:
    # Computes mean difference, percentage difference, Welch 95% CI, Cohen's d, and Hedges' g
    n1, n2 = len(g1), len(g2)
    m1, m2 = float(g1.mean()), float(g2.mean())
    v1, v2 = float(g1.var(ddof=1)), float(g2.var(ddof=1))

    mean_diff = m1 - m2
    pct_diff = (mean_diff / m2) * 100

    # Welch standard error & dof
    se_diff = np.sqrt(v1 / n1 + v2 / n2)
    df = (v1 / n1 + v2 / n2)**2 / (((v1 / n1)**2 / (n1 - 1)) + ((v2 / n2)**2 / (n2 - 1)))
    ci_lower, ci_upper = stats.t.interval(1 - alpha, df=df, loc=mean_diff, scale=se_diff)

    # Pooled standard deviation & Cohen's d
    s_pooled = np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2))
    cohens_d = mean_diff / s_pooled
    hedges_g = cohens_d * (1 - (3 / (4 * (n1 + n2) - 9)))

    return {
        'mean_diff': mean_diff,
        'pct_diff': pct_diff,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        's_pooled': s_pooled,
        'cohens_d': cohens_d,
        'hedges_g': hedges_g
    }

eff = compute_effect_sizes_and_ci(disc_aov, nodisc_aov)

summary_table = pd.DataFrame([
    {'Metric': 'Discounted Mean AOV', 'Value': f"${disc_aov.mean():,.2f}"},
    {'Metric': 'Non-Discounted Mean AOV', 'Value': f"${nodisc_aov.mean():,.2f}"},
    {'Metric': 'Absolute Difference', 'Value': f"+${eff['mean_diff']:,.2f}"},
    {'Metric': 'Percentage Difference', 'Value': f"+{eff['pct_diff']:.2f}%"},
    {'Metric': 'Welch t-statistic', 'Value': f"{welch_res['t_statistic']:.4f}"},
    {'Metric': 'Welch p-value', 'Value': f"{welch_res['p_value']:.4e}"},
    {'Metric': 'Degrees of Freedom', 'Value': f"{welch_res['degrees_of_freedom']:.2f}"},
    {'Metric': 'Mann-Whitney U p-value', 'Value': f"{mwu_p:.4e}"},
    {'Metric': '95% Confidence Interval', 'Value': f"[${eff['ci_lower']:,.2f}, ${eff['ci_upper']:,.2f}]"},
    {'Metric': "Cohen's d (Effect Size)", 'Value': f"{eff['cohens_d']:.4f} (Medium-to-Large)"},
    {'Metric': "Hedges' g (Bias-Corrected)", 'Value': f"{eff['hedges_g']:.4f}"},
    {'Metric': 'Formal Decision', 'Value': 'REJECT H0 (Strong Evidence of Difference)'}
]).set_index('Metric')

print("=== STATISTICAL HYPOTHESIS TESTING REPORT ===")
display(summary_table)

=== STATISTICAL HYPOTHESIS TESTING REPORT ===


,Value
Metric,
Discounted Mean AOV,"$35,550.65"
Non-Discounted Mean AOV,"$18,799.19"
Absolute Difference,"+$16,751.46"
Percentage Difference,+89.11%
Welch t-statistic,13.8758
Welch p-value,1.8665e-30
Degrees of Freedom,184.67
Mann-Whitney U p-value,3.2190e-22
95% Confidence Interval,"[$14,369.70, $19,133.22]"
